
# 🇰🇷 Korean Morphological Analysis — Jupyter Notebook Version
이 노트북은 **Okt 형태소 분석 → 카테고리별 빈도 → 공출현/PMI/PPMI/NPMI → N-gram NPMI → Log-Odds(Dirichlet)**까지 한 번에 실행할 수 있게 구성했습니다.  
모든 CSV는 `UTF-8-SIG`로 저장되고, 시각화는 **matplotlib만** 사용합니다.


In [1]:

# (필요시) 라이브러리 설치
# 주피터 환경에 미설치 시 아래 주석을 해제하고 실행하세요.
# !pip -q install konlpy JPype1 pandas numpy matplotlib


In [2]:

# -*- coding: utf-8 -*-
import os, re, math, json, warnings
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm, rcParams

# 한글 폰트 자동 설정
def set_korean_font():
    candidates = [
        "Malgun Gothic",     # Windows
        "AppleGothic",       # macOS
        "NanumGothic",       # Common
        "Noto Sans CJK KR",  # Noto
        "Noto Sans KR",
    ]
    available = set(f.name for f in fm.fontManager.ttflist)
    for name in candidates:
        if name in available:
            rcParams["font.family"] = name
            break
    rcParams["axes.unicode_minus"] = False

set_korean_font()

# Okt tokenizer
try:
    from konlpy.tag import Okt
except Exception as e:
    raise RuntimeError(
        "konlpy를 불러오지 못했습니다. 상단 설치셀을 실행 후 다시 시도하세요.\n"
        f"원인: {e}"
    )

OKT = Okt()


In [3]:

# ================ 설정 ================
INPUT_CSV = "naver_blog_all_.csv"  # ← 환경에 맞게 변경
OUTPUT_DIR = "morph_outputs"       # 결과 저장 폴더
CATEGORY_COL = ""                             # 예: "카테고리" (없으면 빈 문자열)
ID_COL = ""                                   # 문서 ID 열이 있으면 지정 (선택)

MIN_TOKEN_LEN = 2
POS_KEEP = ("Noun","Adjective","Verb")
MIN_FREQ = 3                  # 최소 빈도(빈도 필터 및 공출현 행렬 규모 제어)
WINDOW = 5                    # 공출현 윈도우
TOP_N = 50                    # 시각화/리포트 참고 상위 토큰 수(필요 시)
NGRAM_MIN_COUNT = 5           # 바이그램/트라이그램 최소 등장수
STOPWORDS_EXTRA_PATH = "/mnt/data/stopwords_extra.txt"  # 추가 불용어 파일(없으면 무시)

os.makedirs(OUTPUT_DIR, exist_ok=True)
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
BASE = os.path.splitext(os.path.basename(INPUT_CSV))[0]


In [4]:

# 기본 불용어 (이전 대화 반영)
DEFAULT_STOPWORDS = set([
    # 조사/접속사/불용어 (일반)
    "이", "그", "저", "것", "거", "수", "등", "때문", "때문에", "및", "그리고", "그러나", "그래서", "또한",
    "으로", "로", "은", "는", "이", "가", "을", "를", "과", "와", "하고", "보다", "에서", "에게", "에도",
    "에는", "이다", "아니다", "되다", "하다", "있다", "없다", "같다", "정도", "부분", "위해", "대한",
    "뿐", "처럼", "같은", "듯", "듯이", "거의", "각각", "모든", "아무", "이런", "그런", "저런", "어떤",
    "무슨", "등등", "부터", "까지", "만", "더", "가장", "제일", "아주", "매우", "너무", "정말", "진짜",
    "그냥", "혹시", "대부분", "여러", "수준",
    # 리뷰/상거래 도메인
    "제품","상품","구성","구입","구매","배송","포장","가격","행사","세트","옵션","용량","맛","향","느낌",
    "사용","효과","후기","리뷰","평가","별점","평점","추천","만족","불만","개선","재구매","성분","브랜드",
    "수량","개","박스","병","캡슐","정","분","알","가루","분말","ml","mg","g","kg","개입","세일",
    "이벤트","증정","사은품","주문","선물","쇼핑","쇼핑백","리뷰수",
    # 동사/형용사(상투어 경향)
    "먹다","먹기","맛있다","좋다","괜찮다","간편하다","간단하다","꾸준하다","편하다","받다","사다","들다",
    "자다","되다","가다","오다","없다","같다","보다","재다","되었다","되어다","요즘",
    # 건강/영양(과잉 일반어)
    "건강","멀티","비타민","알약","섭취","영양제"
])

def load_extra_stopwords(path: str) -> set:
    extra = set()
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                w = line.strip()
                if w and not w.startswith("#"):
                    extra.add(w)
    return extra

RE_URL = re.compile(r"https?://\S+|www\.\S+")
RE_EMAIL = re.compile(r"[\w\.-]+@[\w\.-]+")
RE_MENTION = re.compile(r"[@#][\w\-_]+")
RE_NON_KR = re.compile(r"[^0-9A-Za-z가-힣ㄱ-ㅎㅏ-ㅣ\s]")

def clean_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = RE_URL.sub(" ", s)
    s = RE_EMAIL.sub(" ", s)
    s = RE_MENTION.sub(" ", s)
    s = RE_NON_KR.sub(" ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def detect_text_col(df: pd.DataFrame, candidates=("review_text","리뷰","본문","content","text","댓글","comment","body","post")) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    for c in df.columns:
        if df[c].dtype == "object":
            return c
    raise ValueError("텍스트 컬럼을 찾지 못했습니다. 후보명을 확인하세요.")


In [5]:

def tokenize(text: str, stopwords:set, pos_keep=POS_KEEP, min_len:int=MIN_TOKEN_LEN) -> List[str]:
    text = clean_text(text)
    if not text:
        return []
    pairs = OKT.pos(text, norm=True, stem=True)
    toks = [w for w,p in pairs if (p in pos_keep and len(w) >= min_len and w not in stopwords)]
    return toks

def docs_from_series(series: pd.Series, stopwords:set) -> List[List[str]]:
    docs = []
    for s in series.fillna("").astype(str):
        docs.append(tokenize(s, stopwords))
    return docs

def freq(docs: List[List[str]]) -> Counter:
    c = Counter()
    for d in docs:
        c.update(d)
    return c


In [6]:

def cooc_counts(docs: List[List[str]], vocab: Optional[Dict[str,int]]=None, window:int=WINDOW):
    if vocab is None:
        vocab = {}
        for d in docs:
            for w in d:
                if w not in vocab:
                    vocab[w] = len(vocab)
    V = len(vocab)
    mat = np.zeros((V, V), dtype=np.float64)
    for d in docs:
        L = len(d)
        for i, w in enumerate(d):
            wi = vocab[w]
            j_start = max(0, i - window)
            j_end = min(L, i + window + 1)
            for j in range(j_start, j_end):
                if i == j: 
                    continue
                wj = vocab[d[j]]
                mat[wi, wj] += 1.0
    return mat, vocab

def pmi_matrices(cooc: np.ndarray, eps:float=1e-12):
    total = cooc.sum() + eps
    Pij = cooc / total
    Pi = Pij.sum(axis=1, keepdims=True) + eps
    Pj = Pij.sum(axis=0, keepdims=True) + eps
    PMI = np.log((Pij + eps) / (Pi @ Pj))
    PPMI = np.maximum(PMI, 0.0)
    NPMI = PMI / (-np.log(Pij + eps))
    return PMI, PPMI, NPMI

def ngram_npmi(docs: List[List[str]], n:int=2, min_count:int=NGRAM_MIN_COUNT) -> pd.DataFrame:
    ngram_counts = Counter()
    unigram_counts = Counter()
    total_tokens = 0
    for d in docs:
        for w in d:
            unigram_counts[w] += 1
            total_tokens += 1
        if len(d) >= n:
            for i in range(len(d)-n+1):
                ngram = tuple(d[i:i+n])
                ngram_counts[ngram] += 1
    rows = []
    total_ngrams = sum(ngram_counts.values())
    for ng, c in ngram_counts.items():
        if c < min_count:
            continue
        p_ng = c / max(total_ngrams, 1)
        p_prod = 1.0
        for w in ng:
            p_prod *= (unigram_counts[w] / max(total_tokens,1))
        pmi = math.log((p_ng + 1e-12) / (p_prod + 1e-12))
        npmi = pmi / (-math.log(p_ng + 1e-12))
        rows.append({"ngram":" ".join(ng), "count":c, "PMI":pmi, "NPMI":npmi})
    df = pd.DataFrame(rows).sort_values("NPMI", ascending=False)
    return df


In [7]:

def log_odds_dirichlet(cat_counts: Dict[str, Counter], alpha: float = 0.1) -> pd.DataFrame:
    cats = sorted(cat_counts.keys())
    if len(cats) < 2:
        return pd.DataFrame()
    vocab = set()
    for c in cats:
        vocab |= set(cat_counts[c].keys())
    vocab = sorted(vocab)
    bg = Counter()
    for c in cats:
        bg.update(cat_counts[c])
    bg_total = sum(bg.values())
    alpha_w = {w: alpha * (bg[w] / max(bg_total,1)) for w in vocab}

    rows = []
    for i in range(len(cats)):
        for j in range(i+1, len(cats)):
            ci, cj = cats[i], cats[j]
            Ni = sum(cat_counts[ci].values())
            Nj = sum(cat_counts[cj].values())
            for w in vocab:
                xi = cat_counts[ci][w]
                xj = cat_counts[cj][w]
                pi = (xi + alpha_w[w]) / (Ni + alpha)
                pj = (xj + alpha_w[w]) / (Nj + alpha)
                li = math.log(pi / (1-pi + 1e-12) + 1e-12)
                lj = math.log(pj / (1-pj + 1e-12) + 1e-12)
                delta = li - lj
                var = 1.0/(xi + alpha_w[w]) + 1.0/(xj + alpha_w[w])
                z = delta / math.sqrt(var + 1e-12)
                rows.append({"word": w, "cat_a": ci, "cat_b": cj, "z": z, "delta": delta, "xi": xi, "xj": xj})
    df = pd.DataFrame(rows).sort_values("z", ascending=False)
    return df


In [8]:

def plot_bar_top(freq_df: pd.DataFrame, title: str, out_png: str, top_n:int=10):
    set_korean_font()
    df = freq_df.head(top_n)
    plt.figure(figsize=(8,5))
    plt.bar(df["token"], df["freq"])
    plt.title(title)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.show()

def plot_heatmap(mat: np.ndarray, vocab_list: List[str], out_png: str, top_n:int=30):
    set_korean_font()
    sums = mat.sum(axis=1)
    idx = np.argsort(-sums)[:top_n]
    sub = mat[np.ix_(idx, idx)]
    labels = [vocab_list[i] for i in idx]
    plt.figure(figsize=(8,7))
    plt.imshow(sub, interpolation="nearest", aspect="auto")
    plt.title("Co-occurrence (counts) - top{}".format(top_n))
    plt.xticks(range(len(labels)), labels, rotation=90)
    plt.yticks(range(len(labels)), labels)
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(out_png, dpi=220)
    plt.show()


In [9]:

# 데이터 불러오기 (cp949 폴백)
try:
    df = pd.read_csv(INPUT_CSV, encoding="utf-8", low_memory=False)
except Exception:
    df = pd.read_csv(INPUT_CSV, encoding="cp949", low_memory=False)

text_col = detect_text_col(df)
cat_col = CATEGORY_COL if (CATEGORY_COL and CATEGORY_COL in df.columns) else None
if not cat_col:
    for c in ["카테고리","category","Category","cate","분류","대분류","소분류"]:
        if c in df.columns:
            cat_col = c
            break

print("텍스트 컬럼:", text_col)
print("카테고리 컬럼:", cat_col)
df.head(3)


텍스트 컬럼: keyword
카테고리 컬럼: None


,keyword,title,link,description,bloggername,bloggerlink,postdate
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,도시락 100,광주<b>도시락</b>배달 샐러드로 다이어트까지,https://blog.naver.com/wlgoehdehd/224036526901,"처음에는 <b>도시락</b> 직접 싸가기도 했지만 아침마다 정신없고, 배달음식은 맛...",Love Pop Music Talk (L.P.M.T),blog.naver.com/wlgoehdehd,20251010.0
2,도시락 100,마카오 이심 <b>도시락</b>이심 추천 갤럭시 사용법 할인코드,https://blog.naver.com/19840602/224040238336,<b>도시락</b> 이심 eSIM 사용이었어요. 예전엔 유심을 바꿔 끼우느라 분실 ...,꽃맘이간다,blog.naver.com/19840602,20251014.0


In [10]:

stopwords = DEFAULT_STOPWORDS | load_extra_stopwords(STOPWORDS_EXTRA_PATH)
docs_raw = docs_from_series(df[text_col], stopwords)

# 토큰 저장
df_tokens = df.copy()
df_tokens["__tokens__"] = [" ".join(d) for d in docs_raw]
out_tokens = os.path.join(OUTPUT_DIR, f"{BASE}_tokens_{STAMP}.csv")
df_tokens.to_csv(out_tokens, index=False, encoding="utf-8-sig")
print("토큰 CSV 저장:", out_tokens)
df_tokens.head(3)


토큰 CSV 저장: morph_outputs\naver_blog_all__tokens_20251023_212225.csv


,keyword,title,link,description,bloggername,bloggerlink,postdate,__tokens__
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
1,도시락 100,광주<b>도시락</b>배달 샐러드로 다이어트까지,https://blog.naver.com/wlgoehdehd/224036526901,"처음에는 <b>도시락</b> 직접 싸가기도 했지만 아침마다 정신없고, 배달음식은 맛...",Love Pop Music Talk (L.P.M.T),blog.naver.com/wlgoehdehd,20251010.0,도시락
2,도시락 100,마카오 이심 <b>도시락</b>이심 추천 갤럭시 사용법 할인코드,https://blog.naver.com/19840602/224040238336,<b>도시락</b> 이심 eSIM 사용이었어요. 예전엔 유심을 바꿔 끼우느라 분실 ...,꽃맘이간다,blog.naver.com/19840602,20251014.0,도시락


In [11]:

# 전체 빈도
freq_all = freq(docs_raw)
freq_df = pd.DataFrame([{"token":k, "freq":v} for k,v in freq_all.items()])
freq_df = freq_df[freq_df["freq"] >= MIN_FREQ].sort_values("freq", ascending=False)
out_freq_all = os.path.join(OUTPUT_DIR, f"{BASE}_freq_overall_{STAMP}.csv")
freq_df.to_csv(out_freq_all, index=False, encoding="utf-8-sig")
print("전체 빈도 CSV:", out_freq_all)

# 카테고리별 빈도
if cat_col:
    rows = []
    by_cat = df_tokens.groupby(cat_col)["__tokens__"]
    for cat, s in by_cat:
        sub_docs = [x.split() for x in s.fillna("").tolist()]
        c = freq(sub_docs)
        for k,v in c.items():
            if v >= MIN_FREQ:
                rows.append({"category": cat, "token": k, "freq": v})
    freq_cat_df = pd.DataFrame(rows).sort_values(["category","freq"], ascending=[True,False])
    out_freq_cat = os.path.join(OUTPUT_DIR, f"{BASE}_freq_by_category_{STAMP}.csv")
    freq_cat_df.to_csv(out_freq_cat, index=False, encoding="utf-8-sig")
    print("카테고리별 빈도 CSV:", out_freq_cat)
else:
    freq_cat_df = pd.DataFrame()
    print("카테고리 열이 없어 per-category 빈도는 건너뜀.")

freq_df.head(10)


전체 빈도 CSV: morph_outputs\naver_blog_all__freq_overall_20251023_212225.csv
카테고리 열이 없어 per-category 빈도는 건너뜀.


,token,freq
0,도시락,100
1,양제,100
2,직장,100
3,점심,100
4,루틴,100
5,마켓,100
6,컬리,100
7,뷰티,100


In [12]:

# 공출현 행렬 규모 제한: MIN_FREQ 이상 등장한 토큰만 사용
vocab_keep = set(freq_df["token"].tolist())
docs = [[w for w in d if w in vocab_keep] for d in docs_raw]


In [13]:

cooc, vocab = cooc_counts(docs, window=WINDOW)
vocab_list = [""] * len(vocab)
for w,i in vocab.items():
    vocab_list[i] = w

out_cooc = os.path.join(OUTPUT_DIR, f"{BASE}_cooc_counts_{STAMP}.csv")
pd.DataFrame(cooc, index=vocab_list, columns=vocab_list).to_csv(out_cooc, encoding="utf-8-sig")
print("공출현 행렬 CSV:", out_cooc)

PMI, PPMI, NPMI = pmi_matrices(cooc)
pd.DataFrame(PMI, index=vocab_list, columns=vocab_list).to_csv(
    os.path.join(OUTPUT_DIR, f"{BASE}_pmi_{STAMP}.csv"), encoding="utf-8-sig")
pd.DataFrame(PPMI, index=vocab_list, columns=vocab_list).to_csv(
    os.path.join(OUTPUT_DIR, f"{BASE}_ppmi_{STAMP}.csv"), encoding="utf-8-sig")
pd.DataFrame(NPMI, index=vocab_list, columns=vocab_list).to_csv(
    os.path.join(OUTPUT_DIR, f"{BASE}_npmi_{STAMP}.csv"), encoding="utf-8-sig")
print("PMI/PPMI/NPMI CSV 저장 완료.")


공출현 행렬 CSV: morph_outputs\naver_blog_all__cooc_counts_20251023_212225.csv
PMI/PPMI/NPMI CSV 저장 완료.


In [14]:

bigr = ngram_npmi(docs, n=2, min_count=NGRAM_MIN_COUNT)
trigr = ngram_npmi(docs, n=3, min_count=NGRAM_MIN_COUNT)
bigr_path = os.path.join(OUTPUT_DIR, f"{BASE}_bigrams_npmi_{STAMP}.csv")
trigr_path = os.path.join(OUTPUT_DIR, f"{BASE}_trigrams_npmi_{STAMP}.csv")
bigr.to_csv(bigr_path, index=False, encoding="utf-8-sig")
trigr.to_csv(trigr_path, index=False, encoding="utf-8-sig")
print("바이그램/트라이그램 NPMI CSV:", bigr_path, trigr_path)

bigr.head(10)


KeyError: 'NPMI'

In [ ]:

if cat_col:
    cat_counts = {}
    for cat, s in df_tokens.groupby(cat_col)["__tokens__"]:
        sub_docs = [x.split() for x in s.fillna("").tolist()]
        cat_counts[cat] = freq(sub_docs)
    logodds = log_odds_dirichlet(cat_counts, alpha=0.1)
    out_logodds = os.path.join(OUTPUT_DIR, f"{BASE}_logodds_dirichlet_{STAMP}.csv")
    logodds.to_csv(out_logodds, index=False, encoding="utf-8-sig")
    print("Log-Odds CSV:", out_logodds)
    logodds.head(10)
else:
    print("카테고리 열이 없어 Log-Odds 분석 건너뜀.")


In [ ]:

try:
    top_png = os.path.join(OUTPUT_DIR, f"{BASE}_top10_{STAMP}.png")
    plot_bar_top(freq_df, f"상위 토큰 Top10 ({BASE})", top_png, top_n=10)
    print("Top10 막대 저장:", top_png)

    heat_png = os.path.join(OUTPUT_DIR, f"{BASE}_cooc_heatmap_{STAMP}.png")
    plot_heatmap(cooc, vocab_list, heat_png, top_n=30)
    print("공출현 히트맵 저장:", heat_png)
except Exception as e:
    warnings.warn(f"플롯 생성 경고: {e}")
